# 🧠 MindPulse: Predicting Depression, Anxiety, and Stress (DASS-21)
### Machine Learning & Multi-Task Deep Learning Pipeline on Real Psychometric Data
**Author:** Megha Bhandari  
**Skills & Tech Stack:** Python, Scikit-learn, TensorFlow / Keras, Pandas, NumPy, Matplotlib  
**Dataset:** OpenPsychometrics DASS Research Dataset ($N = 39,775$, validated $N = 6,000$)

---
## 1. Project Overview & Clinical Framework
This notebook implements an end-to-end clinical psychometric prediction system based on the **Depression, Anxiety, and Stress Scales (DASS-21)** developed by the University of New South Wales (Lovibond & Lovibond).

Instead of synthetic formulas, this project trains on **authentic human survey data** from the OpenPsychometrics research archive. We benchmark:
1. **Scikit-learn Supervised Classifiers**: Random Forest, Gradient Boosting, and Multinomial Logistic Regression.
2. **Multi-Task Deep Neural Network (TensorFlow/Keras)**: Joint latent feature extraction with three task heads predicting Depression, Anxiety, and Stress severity levels simultaneously.

In [1]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

from src.data_loader import load_psychometric_dataset, SEVERITY_LEVELS
from src.preprocessing import DASSFeatureEngineer

print('All libraries successfully imported!')

## 2. Load Real-World Psychometric Dataset
We load real respondent data from `data/real_dass_data.csv` (OpenPsychometrics archive, filtered for valid completion latencies and mapped to standard DASS-21 0–3 Likert scale).

In [2]:
df = load_psychometric_dataset(n_samples=6000, random_state=42)
print('Dataset Shape:', df.shape)
df[['Q1', 'Q2', 'Q3', 'dep_sum', 'anx_sum', 'str_sum', 'Depression_Class', 'Anxiety_Class', 'Stress_Class']].head()

## 3. Exploratory Data Analysis (EDA) with Matplotlib
Visualizing distribution of clinical subscale scores across participants.

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
subscales = [
    ('dep_sum', 'Depression Score Distribution', '#EF4444'),
    ('anx_sum', 'Anxiety Score Distribution', '#F59E0B'),
    ('str_sum', 'Stress Score Distribution', '#3B82F6')
]
for ax, (col, title, color) in zip(axes, subscales):
    ax.hist(df[col], bins=20, color=color, edgecolor='#1E293B', alpha=0.85)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('DASS-42 Equivalent Score')
    ax.set_ylabel('Participant Count')
    ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Feature Engineering Pipeline (Scikit-learn Transformer)
Extract composite distress indicators, autonomic arousal clusters, anhedonia indices, and cross-subscale ratios.

In [4]:
feat_eng = DASSFeatureEngineer()
X_raw = df[[f'Q{i+1}' for i in range(21)]]
X_engineered = feat_eng.fit_transform(X_raw)
feature_names = feat_eng.get_feature_names_out()
print('Raw Feature Count:', X_raw.shape[1])
print('Engineered Feature Count:', X_engineered.shape[1])
print('Extracted Features:', feature_names)

## 5. Supervised Machine Learning Benchmarking
Evaluating Random Forest and Gradient Boosting on Depression severity classification.

In [5]:
y_dep = df['Depression_Class']
X_tr, X_te, y_tr, y_te = train_test_split(X_engineered, y_dep, test_size=0.2, random_state=42, stratify=y_dep)
rf = RandomForestClassifier(n_estimators=120, max_depth=12, random_state=42)
rf.fit(X_tr, y_tr)
rf_preds = rf.predict(X_te)
print('Random Forest Accuracy:', round(accuracy_score(y_te, rf_preds)*100, 2), '%')
print('\nClassification Report:')
print(classification_report(y_te, rf_preds, zero_division=0))

## 6. Multi-Task Deep Neural Network (TensorFlow/Keras)
Testing our trained Multi-Task DNN model (`models/multitask_dass_model.keras`) that shares latent representations across all three clinical tasks.

In [6]:
import tensorflow as tf
import pickle
deep_model = tf.keras.models.load_model('../models/multitask_dass_model.keras')
with open('../models/preprocessor.pkl', 'rb') as f:
    scaler = pickle.load(f)['scaler']
with open('../models/label_encoder.pkl', 'rb') as f:
    classes = pickle.load(f)
X_scaled = scaler.transform(X_engineered[:5])
preds = deep_model.predict(X_scaled)
print('Multi-Task DNN Prediction Heads: Depression, Anxiety, Stress')
for i in range(5):
    d_cls = classes[np.argmax(preds[0][i])]
    a_cls = classes[np.argmax(preds[1][i])]
    s_cls = classes[np.argmax(preds[2][i])]
    print(f'Participant {i+1} -> Dep: {d_cls} ({np.max(preds[0][i])*100:.1f}%), Anx: {a_cls} ({np.max(preds[1][i])*100:.1f}%), Str: {s_cls} ({np.max(preds[2][i])*100:.1f}%)')